```
- Copyright 2023 DeepMind Technologies Limited
- All software is licensed under the Apache License, Version 2.0 (Apache 2.0); you may not use this file except in compliance with the Apache 2.0 license. You may obtain a copy of the Apache 2.0 license at: https://www.apache.org/licenses/LICENSE-2.0
- All other materials are licensed under the Creative Commons Attribution 4.0 International License (CC-BY).  You may obtain a copy of the CC-BY license at: https://creativecommons.org/licenses/by/4.0/legalcode
- Unless required by applicable law or agreed to in writing, all software and materials distributed here under the Apache 2.0 or CC-BY licenses are distributed on an \"AS IS\" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the licenses for the specific language governing permissions and limitations under those licenses.
- This is not an official Google product
```

# Cyclic graphs

This notebook contains:
1. the *skeleton* we used for X-evolve to discover large independent sets in the $n$-th strong product of cyclic graphs,
2. the *prompt* we used for this problem to instruct large language model,
3. the *functions* discovered by X-evolve that construct those independent sets.

## Skeleton

The *skeleton* we used for X-evolve is from [FunSearch](https://github.com/google-deepmind/funsearch).


In [ ]:
"""Obtains maximal independent sets."""
import itertools
import numpy as np


# @funsearch.run
def evaluate(num_nodes: int, n: int) -> int:
  """Returns the size of an independent set."""
  independent_set = solve(num_nodes, n)
  return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
  """Gets independent set with maximal size.

  Args:
    num_nodes: The number of nodes of the base cyclic graph.
    n: The power we raise the graph to.

  Returns:
    A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
  """
  to_block = np.array(list(itertools.product([-1, 0, 1], repeat=n)))

  # Powers in decreasing order for compatibility with `itertools.product`, so
  # that the relationship `i = children[i] @ powers` holds for all `i`.
  powers = num_nodes ** np.arange(n - 1, -1, -1)

  # Precompute the priority scores.
  children = np.array(
      list(itertools.product(range(num_nodes), repeat=n)), dtype=np.int32)
  scores = np.array([priority(tuple(child), num_nodes, n)
                     for child in children])

  # Build `max_set` greedily, using scores for prioritization.
  max_set = np.empty(shape=(0, n), dtype=np.int32)
  while np.any(scores != -np.inf):
    # Add a child with a maximum score to `max_set`, and set scores of
    # invalidated children to -inf, so that they never get selected.
    max_index = np.argmax(scores)
    child = children[None, max_index]  # [1, n]

    blocking = np.einsum(
        'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
    scores[blocking] = -np.inf
    max_set = np.concatenate([max_set, child], axis=0)

  return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
  """Returns the priority with which we want to add `el` to the set.

  Args:
    el: an n-tuple representing the element to consider whether to add.
    num_nodes: the number of nodes of the base graph.
    n: an integer, power of the graph.

  Returns:
    A number reflecting the priority with which we want to add `el` to the
    independent set.
  """
  return 0.

## Prompt

The *prompt* we used for X-evolve includes an initial program.

The initial program is the $\mathcal{C}_9$ priority function provided by [Funsearch](https://github.com/google-deepmind/funsearch), but for the Shannon capacity problem in $\mathcal{C}_9$ the initial program is the $\mathcal{C}_7$ priority function provided by [Funsearch](https://github.com/google-deepmind/funsearch).


In [ ]:
f'''I'm working on the maximum independent set problem in the {n_dim}-th strong product of a {num_nodes}-node cycle graph, using a greedy algorithm guided by a priority function to determine vector selection order.Each vertex in this graph is a {n_dim}-dimensional vector with values in {0, 1, ..., num_nodes-1}. Two vertices are adjacent if they differ by at most 1 (mod {num_nodes}) in each coordinate and are not identical. An independent set is a subset of vectors where no two are adjacent.


## What I Need
1. **BOLD EVOLUTION OF PRIORITY FUNCTION**: Please create an improved `priority_new` function that might outperform my reference implementations. Don't be constrained by my current approaches - take risks and suggest radically different strategies that might lead to breakthroughs.
2. **MARK ALL TUNABLE PARAMETERS**: For every element in the `priority_new` function that could potentially be tuned, wrap it with tunable([option1, option2, ...]).
  Format examples:
    - `if x == tunable([x1, x2, x3]):`
    - `z = tunable([x + y, x * (y + 1)])`


## Task Description
Please help me develop an improved `priority_new` function by analyzing my reference implementations.
Output Python code only, without any comments.
The function should be concise.


## Current Priority Functions
Below are two reference priority functions I've developed.

import itertools
import numpy as np
import pickle

@funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs{nodes_dim}.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


@funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    num_nodes = {num_nodes}
    n = {n_dim}
    return 0.
'''

## Discovered functions that respectively build the best known independent sets in $C_9^n$ for $n=3,...,7$

In [2]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs9_3.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    num_nodes = 9
    n = 3
    score = 0.
    for i in range(n):
        if el[i] == el[(i + 1) % n]:
            score += 2
        else:
            score -= 1
        x = (n * el[i] - el[(i + 1) % n] - el[(i + 2) % n] - (n + 1) * el[(i + 3) % n]) % num_nodes
        score -= 0.7 * (x - el[(i + 1) % n]) ** 2
        score += 0.3 * (num_nodes - 1 - (x - 1) % num_nodes) ** 2
        score += 0.3 * (num_nodes - 1 - (x - 2) % num_nodes) ** 3
    return score

print(evaluate(dict(num_nodes=9, n=3)))

81


In [3]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs9_4.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    num_nodes = 9
    n = 4
    score = 0.5
    for i in range(n // 2):
        neighbor_count = sum(1 for j in range(n) if abs((el[i] - el[j]) % num_nodes) <= 2)
        score += 0.1 * (el[i] == el[(i + 2) % n])
        score -= 1.0 * neighbor_count
        x = ((n-1 * el[i] - el[(i + 1) % n] - el[(i + 3) % n] - n * el[(i + 3) % n]) % num_nodes)
        score += 0.01 * ((x - el[(i + 1) % n]) % num_nodes) ** 3
        score -= 0.001 * (num_nodes - 2 - (x - 2) % num_nodes) ** 1
    return score

print(evaluate(dict(num_nodes=9, n=4)))

324


In [4]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs9_5.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    num_nodes = 9
    n = 5
    score = 5.
    for i in range(n // 2):
        if False:
            neighbor_count = sum(1 for j in range(n) if abs((el[i] - el[j]) % num_nodes) <= 2)
        else:
            neighbor_count = n
        score += 0.1 * (1 if el[i] == el[(i + 3) % n] else -1)
        x = ((n * el[i] - el[(i + 2) % n] - el[(i + 3) % n]) % num_nodes)
        score -= 1. * (x - el[(i + 2) % n]) ** 2
        score += 0.1 * (num_nodes - neighbor_count) ** 1
    return score

print(evaluate(dict(num_nodes=9, n=5)))

1458


In [5]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs9_6.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    num_nodes = 9
    n = 6
    score = 0.
    for i in range(n):
        if el[i] == el[(i + 2) % n]:
            score += 2
        else:
            score -= 2
        x = ((n - 2) * el[i] - el[(i + 2) % n] - el[(i + 2) % n] - (n + 2) * el[(i + 4) % n]) % num_nodes
        score -= 0.2 * (x - el[(i + 1) % n]) ** 3
        score += 0.3 * (num_nodes - 2 - (x - 2) % num_nodes) ** 3
        score += 0.3 * (num_nodes - 1 - (x - 3) % num_nodes) ** 3
    return score

print(evaluate(dict(num_nodes=9, n=6)))

6561


In [6]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs9_7.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    num_nodes = 9
    n = 7
    score = 0.0
    for i in range(n // 3):
        if el[i] == el[(i + 2) % n]:
            score += 1
        else:
            score -= 0.5
        x = ((n - 3 * el[i] - el[(i + 5) % n] - el[(i + 7) % n] - (n) * el[(i + 2) % n]) % num_nodes)
        score -= 0.2 * (x - el[(i + 3) % n]) ** 1
        score += 0.2 * (num_nodes - 1 - (x - 2) % num_nodes) ** 3
    score += 0.1 * sum((el[i] - el[(i + 2) % n]) ** 2 for i in range(n))
    score *= 1
    return score

print(evaluate(dict(num_nodes=9, n=7)))

26244


## Discoverd function that builds the best known independent set of size 148 in $\mathcal{C}_{11}^{3}$

In [4]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs11_3.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    base = 17
    n = 3
    s = -1.
    for i in range(n):
        s += 2. * (el[i] << i) 
        s %= base+1
    return (3. * el[2] 
            + -2. * el[2]) % base + s
    

print(evaluate(dict(num_nodes=11, n=3)))

148


## Discoverd function that builds the best known independent set of size 754 in $\mathcal{C}_{11}^{4}$

In [1]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs11_4.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    tuned = 19
    n_tuned = 5
    s = 0
    weight = 0.5
    exponent = 2
    for i in range(n_tuned):
        s += el[i % len(el)] * (exponent ** i)
        s %= tuned - 1
    avg_el = sum(el) / len(el)
    diff = max(el) - min(el)
    return ((-1 * el[-1] + 
             -1 * el[0]) % tuned + 
            -tuned + 
            weight * s + 
            -1 * avg_el + 
            0.1 * diff + 
            1 * (el[-1] - el[0]))
    

print(evaluate(dict(num_nodes=11, n=4)))

754


## Discoverd function that builds the best known independent set of size 247 in $\mathcal{C}_{13}^{3}$

In [6]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs13_3.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    num_nodes = 13
    n = 3
    s = 0.
    for i in range(n):
        s += el[i] << i + 4
        s %= num_nodes
    return (6 * el[2] + 1 * el[0] + 3 * el[1] + 0) % num_nodes + s
   

print(evaluate(dict(num_nodes=13, n=3)))

247


## Discoverd function that builds the best known independent set of size 9633 in $\mathcal{C}_{13}^{5}$

In [2]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs13_5.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    weight = 13
    width = 3
    s = 1.0
    for i in range(width):
        s += el[i % len(el)] * i + 11
        s %= weight
    return (7 * el[0] + 
            10 * el[3] + 
            1 * el[len(el) - 1]) % weight // 5 + s
    

print(evaluate(dict(num_nodes=13, n=5)))

9633


## Discovered function that finds an independent set of size 19946 in $C_{15}^5$

This is larger than the best known independent set reported by [David de Boer, Pjotr Buys & Jeroen Zuiddam(2024)](https://arxiv.org/pdf/2404.16763), which has size $19894$.

In [1]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs15_5.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    base_num_nodes = 13
    weight = 1.5
    s = 25.0
    for i in range(n):
        s += weight * el[i % n] * 2 ** i
        s %= base_num_nodes
    factors = [
        6 * el[4], 
        3 * el[3], 
        1 * el[2], 
        0.08 * el[1], 
        0.04 * el[0]
    ]
    combined_factor = sum(factors)
    dynamic_weight = 2.0
    additional_offset = 2.0
    return (combined_factor + additional_offset) % base_num_nodes + s * dynamic_weight

print(evaluate(dict(num_nodes=15, n=5)))

19946


## Other discovered functions in $C_{15}^5$

In [ ]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs15_5.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    base = 13
    weight = 1.5
    s = 25.0
    for i in range(n):
        s += weight * el[i % n] * 2 ** i
        s %= base
    factors = [
        6 * el[4], 
        3 * el[3], 
        1 * el[2], 
        0.06 * el[1], 
        0.03 * el[0]
    ]
    combined_factor = sum(factors)
    dynamic_weight = 2.5
    additional_offset = 0.5
    return (combined_factor + additional_offset) % base + s * dynamic_weight

print(evaluate(dict(num_nodes=15, n=5)))

19945


In [4]:
import itertools
import numpy as np
import pickle

# @funsearch.run
def evaluate(params: dict) -> int:
    """Returns the size of an independent set."""
    independent_set = solve(params['num_nodes'], params['n'])
    return len(independent_set)


def solve(num_nodes: int, n: int) -> list[tuple[int, ...]]:
    """Gets independent set with maximal size.

    Args:
        num_nodes: The number of nodes of the base cyclic graph.
        n: The power we raise the graph to.

    Returns:
        A list of `n`-tuples in `{0, 1, 2, ..., num_nodes - 1}`.
    """
    with open('cycle_graphs15_5.pkl', 'rb') as f:
        to_block, powers, children = pickle.load(f)
    scores = np.array([priority(tuple(child), num_nodes, n)
                        for child in children],dtype=np.float32)

    # Build `max_set` greedily, using scores for prioritization.
    max_set = np.empty(shape=(0, n), dtype=np.int32)
    while np.any(scores != -np.inf):
        # Add a child with a maximum score to `max_set`, and set scores of
        # invalidated children to -inf, so that they never get selected.
        max_index = np.argmax(scores)
        child = children[None, max_index]  # [1, n]

        blocking = np.einsum(
            'cn,n->c', (to_block + child) % num_nodes, powers)  # [C]
        scores[blocking] = -np.inf
        max_set = np.concatenate([max_set, child], axis=0)

    return [tuple(map(int, el)) for el in max_set]


# @funsearch.evolve
def priority(el: tuple[int, ...], num_nodes: int, n: int) -> float:
    """Returns the priority with which we want to add `el` to the set.

    Args:
        el: an n-tuple representing the element to consider whether to add.
        num_nodes: the number of nodes of the base graph.
        n: an integer, power of the graph.

    Returns:
        A number reflecting the priority with which we want to add `el` to the
        independent set.
    """
    base = 13
    weight = 1.5
    s = 15.0
    for i in range(n):
        s += weight * el[i] * 2 ** i
        s %= base
    factors = [
        6 * el[4], 
        3 * el[3], 
        1 * el[2], 
        0.06 * el[1], 
        0.03 * el[0]
    ]
    combined_factor = sum(factors)
    dynamic_weight = 2.5
    additional_offset = 0.5
    return (combined_factor + additional_offset) % base + s * dynamic_weight

print(evaluate(dict(num_nodes=15, n=5)))

19944
